# 06 · Features from the shared dataset

Recomputes the per-contact features used by `04_statistics_and_figures.ipynb` from the shared dataset, so that
the paper's ERP and theta results can be reproduced without access to the raw recordings.

1. Download and unzip the shared data, and point `SHARED_DATA_DIR` to that folder
   (`export SHARED_DATA_DIR=/path/to/shared_data`; default: `derivatives/shared_data`).
2. Run this notebook (slow: phase clustering is a circular kernel density per contact, frequency and time point).
3. Run `01_behaviour.ipynb` and `04_statistics_and_figures.ipynb` with `USE_SHARED_DATA=1`.
   Results and figures are written to `results/from_shared_data/` and `figures/from_shared_data/`.

Theta power, phase clustering and phase speed are computed with the same code as in
`03_feature_extraction.ipynb` (`features.theta_features`). The shared data are stored in single precision, so the
results match the full pipeline to numerical precision.

Input: `SHARED_DATA_DIR/sub-XX.h5`. Output: `derivatives/features_from_shared_data/Hippocampus_<sub>_<contact>.pkl`.

In [ ]:
import pickle

import h5py
import numpy as np

import config
import features

config.SHARED_FEATURES_DIR.mkdir(parents=True, exist_ok=True)
N_KEEP = int(round((config.ANALYSIS_TMAX - config.EPOCH_TMIN) * config.SFREQ))  # 768 samples: -500 to 1000 ms

In [ ]:
files = sorted(config.SHARED_DATA_DIR.glob("sub-*.h5"))
assert files, f"No shared data files (sub-*.h5) found in {config.SHARED_DATA_DIR}"

for path in files:
    with h5py.File(path, "r") as h5:
        subject = str(h5.attrs["subject"])
        sfreq = float(h5.attrs["sfreq"])
        theta = h5["theta"][:]                       # (trials, contacts, 2, 769), complex
        missing = h5["missing"][:]                   # (trials, contacts)
        contacts = list(h5["contacts/name"].asstr()[:])
        memorability = h5["trials/memorability"][:]
        lfp = h5["lfp"][:] if "lfp" in h5 else None  # (trials, contacts, 768)
    print(f"{subject}: {theta.shape[0]} trials, {len(contacts)} contacts")

    for c, contact in enumerate(contacts):
        contact_features = {
            **features.theta_features(theta[:, c], sfreq, n_keep=N_KEEP),
            "missing": missing[:, c],
            "memorability": memorability,
        }
        if lfp is not None:
            contact_features["lfp"] = lfp[:, c, np.newaxis, :].astype(np.float64)  # (trials, 1, 768)
        with open(config.SHARED_FEATURES_DIR / f"Hippocampus_{subject}_{contact}.pkl", "wb") as f:
            pickle.dump(contact_features, f)